# Master Pipeline Evaluation & Agent Testbench

**Project:** The Price is Right - Autonomous Deal Hunting AI

**Scope:** End-to-end component validation across data ingestion, vector indexing, RAG estimation, local GGUF inference, ensemble scoring, autonomous planning, and UI rendering.

### 1. Environment & Global Configuration

In [1]:
import os
import sys
import time
import json
import re
import logging
import torch
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm
from openai import OpenAI
import chromadb
from huggingface_hub import login
from sklearn.manifold import TSNE

# Force UTF-8 encoding across environments
if sys.platform == "win32" and hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Configure root directory and environment variales
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
sys.path.append(str(ROOT_DIR / "src"))
load_dotenv(ROOT_DIR/".env", override=True)

# Changing Cache location
HF_PROJECT_CACHE = ROOT_DIR / "data" / "hf_cache"
HF_PROJECT_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_PROJECT_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_PROJECT_CACHE / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(HF_PROJECT_CACHE / "transformers")

# Now importing SentenceTransformer and CrossEncoder after cache paths are established
from sentence_transformers import SentenceTransformer, CrossEncoder

# Centralized logging configuration for pipeline observability
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
)
logger = logging.getLogger("PipelineEvaluator")

# Service Configuration
LOCAL_GGUF_URL = os.getenv("LOCAL_GGUF_URL", "http://localhost:8000/v1")
VECTOR_DB_DIR = (str(ROOT_DIR / "data" / "vectorstore"))
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LORA_MODEL_PATH = ROOT_DIR / "data" / "models" / "specialist_lora"
MODEL_CACHE_DIR = str(ROOT_DIR / "data" / "models" / "transformers") 

logger.info(f"Root Directory: {ROOT_DIR}")
logger.info(f"Vectorstore Path: {VECTOR_DB_DIR}")
logger.info(f"Local Inference Endpoint: {LOCAL_GGUF_URL}")
logger.info(f"Local Hugging Face Cache: {HF_PROJECT_CACHE}")

c:\user_personal_files\Projects\llm_engineering\price_is_right_ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-04 03:25:18,479 | INFO     | PipelineEvaluator | Root Directory: c:\user_personal_files\Projects\llm_engineering\price_is_right_ai
2026-09-04 03:25:18,480 | INFO     | PipelineEvaluator | Vectorstore Path: c:\user_personal_files\Projects\llm_engineering\price_is_right_ai\data\vectorstore
2026-09-04 03:25:18,480 | INFO     | PipelineEvaluator | Local Inference Endpoint: http://localhost:11434/v1
2026-09-04 03:25:18,481 | INFO     | PipelineEvaluator | Local Hugging Face Cache: c:\user_personal_files\Projects\llm_engineering\price_is_right_ai\data\hf_cache


### 1. Messaging & Push Notification Verification

In [2]:
# pushover_user = os.getenv('PUSHOVER_USER')
# pushover_token = os.getenv('PUSHOVER_TOKEN')

# if not pushover_user or not pushover_token:
#     logger.warning("Pushover credentials missing. MessagingAgent will fail silently or crash.")
# else:
#     logger.info(f"Pushover User Token detected: {pushover_user[:2]}***")

load_dotenv(override=True)

ntfy_url = os.getenv('NTFY_URL')
ntfy_topic = os.getenv('NTFY_TOPIC')

if not ntfy_url or not ntfy_topic:
    logger.warning("Ntfy credentials missing. MessagingAgent will fail silently or crash.")
else:
    print(f"Ntfy Topic detected: {ntfy_topic[:5]}...")


Ntfy Topic detected: price...


### 2. Local Inference Server Verification (GGUF / Local LLM)

In [4]:
logger.info("Validating local GGUF inference endpoint...")
local_client = OpenAI(base_url=LOCAL_GGUF_URL, api_key=os.getenv("LOCAL_API_KEY", "not-needed"))

try:
    # Test local endpoint connection to ensure the LLM runner is alive
    models_available = local_client.models.list()
    active_model = models_available.data[0].id if models_available.data else "default-gguf"
    logger.info(f"Local inference online. Active Model ID: {active_model}")
except Exception as e:
    logger.warning(f"Local GGUF endpoint check failed: {e}")
    logger.info("Ensure local runner (llama-cpp-python / Ollama / vLLM) is running.")

2026-09-04 03:25:50,420 | INFO     | PipelineEvaluator | Validating local GGUF inference endpoint...
2026-09-04 03:25:54,525 | INFO     | openai._base_client | Retrying request to /models in 0.428493 seconds
2026-09-04 03:25:59,088 | INFO     | openai._base_client | Retrying request to /models in 0.983875 seconds
2026-09-04 03:26:04,168 | WARNING  | PipelineEvaluator | Local GGUF endpoint check failed: Connection error.
2026-09-04 03:26:04,171 | INFO     | PipelineEvaluator | Ensure local runner (llama-cpp-python / Ollama / vLLM) is running.


### 3. Dataset Ingestion & ChromaDB Vectorstore Indexing

In [5]:
from src.core.items import Item #from core.schemas import Item

hf_token = os.getenv("HF_TOKEN")
if hf_token: login(token=hf_token)

LITE_MODE = True
dataset_name = "ed-donner/items_lite" if LITE_MODE else "ed-donner/items_full"

# Downloads and cacches dataset directly inside data/hf_cache
logger.info(f"Loading dataset '{dataset_name}' (cached locally in data/hf_cache)...")
train, val, test = Item.from_hub(dataset_name)
logger.info(f"Dataset loaded successfully: {len(train)} train, {len(val)} val, {len(test)} test items.")

# Initialize the Bi-Encoder for document embedding
logger.info("Loading BAAI/bge-large-en-v1.5 Bi-Encoder...")
encoder = SentenceTransformer('BAAI/bge-large-en-v1.5', device='cuda', cache_folder=MODEL_CACHE_DIR)

db_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = db_client.get_or_create_collection(name="products")

if collection.count() == 0:
    logger.info("Indexing items into ChromaDB using BGE-Large embeddings...")
    batch_size = 64
    for i in tqdm(range(0, len(train), batch_size), desc="Indexing Batches"):
        batch = train[i : i + batch_size]
        docs = [item.summary for item in batch]

        # BGE handles standard documents effectively. Nomalize embeddings for accurate metric search.
        embeddings = encoder.encode(docs, show_progress_bar=False, normalize_embeddings=True).tolist()
        metas = [{"category": item.category, "price": float(item.price)} for item in batch]
        ids = [f"item_{j}" for j in range(i, i + len(batch))]

        collection.add(ids=ids, documents=docs, embeddings=embeddings, metadatas=metas)
else:
    logger.info(f"ChromaDB collection already populated ({collection.count()} records). Skipping indexing.")

2026-09-04 03:30:07,393 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-09-04 03:30:07,411 | WARNING  | huggingface_hub._login | Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-09-04 03:30:07,414 | INFO     | PipelineEvaluator | Loading dataset 'ed-donner/items_lite' (cached locally in data/hf_cache)...
2026-09-04 03:30:07,786 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/datasets/ed-donner/items_lite/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-09-04 03:30:07,843 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ed-donner/items_lite/057ae6b7731e35538897c976e391e76fbb729006/README.md "HTTP/1.1 200 OK"
2026-09-04 03:30:08,197 | INFO     | httpx | HTTP R

#### 3.5. RAG Retrieval Pipeline (Query Rewriting + Cross-Encoder Reranking)

In [16]:
logger.info("Loading BAAI Cross-Encoder for Reranking...")
# The cross-encoder directly scores the (query, document) pair for highly accurate semantic matching
cross_encoder = CrossEncoder('BAAI/bge-reranker-base', device='cuda', cache_folder=MODEL_CACHE_DIR)

gemini_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/", 
    api_key=GEMINI_API_KEY
)

def advanced_rag_retrieval(raw_query: str, top_k=10, rerank_k=3):
    """
    1. Rewrites the query for optimal search density using local LLM.
    2. Retrieves top_k candidates via BAAI bi-encoder vector search.
    3. Reranks the candidates using BAAI cross-encoder.
    """
    # Query Rewriting via LLM
    rewrite_prompt = f"Rewrite this search query to be highly specific for a product database. Return ONLY the new query string: '{raw_query}'"
    try:
        completion = gemini_client.chat.completions.create(
            model="gemini-3.5-flash-lite",
            messages=[{"role": "user", "content": rewrite_prompt}],
            temperature=0.3,
            max_tokens=30
        )
        rewritten_query = completion.choices[0].message.content.strip().strip("'\"")
    except Exception as e:
        logger.warning(f"Query rewriting failed, falling back to raw query: {e}")
        rewritten_query = raw_query

    logger.info(f"Rewritten query: {rewritten_query}")

    # BGE Bi-Encoders recommend adding an instruction prefix for query embeddings
    instruction = "Represent this sentence for searching relevant passages: "
    query_embedding = encoder.encode(instruction + rewritten_query, normalize_embeddings=True).tolist()

    # Dense Retrieval
    search_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    retrieved_docs = search_results["documents"][0]
    if not retrieved_docs:
        return []

    # Reranking(Cross-Encoder)
    # Prepare pairs of (Query, Document) for the Cross-Encoder
    cross_input = [[rewritten_query, doc] for doc in retrieved_docs]
    rerank_scores = cross_encoder.predict(cross_input)

    # Sort documents by cross-encoder score descending
    doc_score_pairs = list(zip(retrieved_docs, rerank_scores))
    doc_score_pairs.sort(key=lambda x: x[1], reverse=True)

    # Return the top 'rerank_k' results
    return [doc for doc, score in doc_score_pairs[:rerank_k]]


logger.info("Testing advanced RAG pipeline...")
sample_query = "quiet clicks 8k wireless performance mouse"
results = advanced_rag_retrieval(sample_query, top_k=5, rerank_k=2)

print(f"\nTop Reranked Results for '{sample_query}':")
for i, res in enumerate(results, 1):
    print(f"{i}. {res}")

2026-09-03 23:40:46,706 | INFO     | PipelineEvaluator | Loading BAAI Cross-Encoder for Reranking...
2026-09-03 23:42:07,366 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-09-03 23:42:07,374 | INFO     | sentence_transformers.base.model | No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
2026-09-03 23:42:07,685 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-09-03 23:42:08,009 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-03 23:42:08,079 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-09-03 23:42:08,449 | INFO     | httpx | HTTP Request: HEAD 


Top Reranked Results for 'quiet clicks 8k wireless performance mouse':
1. Title: Macally Wireless Ergonomic Keyboard & Quiet Click Mouse  
Category: Computer Accessories  
Brand: Macally  
Description: Ergonomic wireless keyboard and quiet‑click mouse for a natural typing and precise scrolling experience.  
Details: Features a split, angled keyboard design, multi‑device Bluetooth pairing, ambidextrous mouse body, adjustable DPI, and 90% reduced click noise.
2. Title: Milk Tea Wireless Keyboard & Mouse Set  
Category: Electronics  
Brand: FOPETT  
Description: A 104‑key wireless keyboard and silent mouse combo with 2.4 GHz connectivity for PC, laptop, TV, and more.  
Details: Features a 10‑meter range, single Nano receiver, adjustable DPI up to 1600, tilt‑angle keyboard design, and AA battery power.


### 4. Vector Space Inspection: 2D & 3D t-SNE

In [17]:
MAX_VIS_POINTS = 5000
sample_data = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAX_VIS_POINTS)

if len(sample_data['embeddings']) > 0:
    raw_vectors = np.array(sample_data['embeddings'])
    categories = [meta['category'] for meta in sample_data['metadatas']]
    unique_cats = list(set(categories))
    cat_to_idx = {cat: idx for idx, cat in enumerate(unique_cats)}
    colors = [cat_to_idx[c] for c in categories]
    hover_text = [f"Category: {c}<br>Text: {d[:60]}..." for c, d in zip(categories, sample_data['documents'])]

    # 2D projection
    reduced_2d = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(raw_vectors)
    fig_2d = go.Figure(data=[go.Scatter(
        x = reduced_2d[:, 0], y = reduced_2d[:, 1], mode='markers',
        marker=dict(size=4, color=colors, colorscale='Viridis', opacity=0.7),
        text=hover_text, hoverinfo='text'
    )])
    fig_2d.update_layout(title="2D Vectorstore Projection", width=950, height=500)
    fig_2d.show()

    # 3D projection
    reduced_3d = TSNE(n_components=3, random_state=42, perplexity=30).fit_transform(raw_vectors)
    fig_3d = go.Figure(data=[go.Scatter3d(
        x = reduced_3d[:, 0], y = reduced_3d[:, 1], z = reduced_3d[:, 2], mode='markers',
        marker=dict(size=2, color=colors, colorscale='Viridis', opacity=0.7),
        text=hover_text, hoverinfo='text'
    )])
    fig_3d.update_layout(title="3D Vectorstore Projection", width=950, height=600)
    fig_3d.show()


### 5. Rate-Limited Multi-Model Valuation & Ensemble Benchmark

#### 5.1 Setup and Initialization

In [ ]:
# import importlib
# import src.agents.specialist_agent
# import src.agents.frontier_agent
# import src.agents.neural_network_agent
# import src.agents.preprocessor_agent
# import src.agents.ensemble_agent


# importlib.reload(src.agents.specialist_agent)
# importlib.reload(src.agents.frontier_agent)
# importlib.reload(src.agents.neural_network_agent)
# importlib.reload(src.agents.preprocessor_agent)
# importlib.reload(src.agents.ensemble_agent)


<module 'src.agents.ensemble_agent' from 'c:\\user_personal_files\\Projects\\llm_engineering\\price_is_right_ai\\src\\agents\\ensemble_agent.py'>

In [6]:
from dataclasses import dataclass
from src.evaluation.benchmark import evaluate
from src.agents.specialist_agent import SpecialistAgent
from src.agents.frontier_agent import FrontierAgent
from src.agents.neural_network_agent import NeuralNetworkAgent
from src.agents.ensemble_agent import EnsembleAgent
from src.agents.preprocessor_agent import PreprocessorAgent

# Initialize the Sub-Agents
print("Initializing pricing agents...")
specialist = SpecialistAgent()
frontier = FrontierAgent(api_key=GEMINI_API_KEY)
weights_path = ROOT_DIR / "data" / "models" / "deep_neural_network.pth"
nn_agent = NeuralNetworkAgent(weights_path=str(weights_path))
preprocessor = PreprocessorAgent()

print("Ready. ")

2026-09-04 03:30:50,541 | INFO     | Specialist Agent | [Specialist Agent] Specialist Agent initialized: targeting local Ollama (specialist-pricer)
2026-09-04 03:30:50,541 | INFO     | Frontier Agent | [Frontier Agent] Initializing Frontier Agent...
2026-09-04 03:30:50,622 | INFO     | Frontier Agent | [Frontier Agent] Loading BAAI Bi-Encoder globally on cuda...


Initializing pricing agents...


2026-09-04 03:32:11,261 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-04 03:32:11,337 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-09-04 03:32:11,653 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-04 03:32:12,111 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-09-04 03:32:12,114 | INFO     | sentence_transformers.base.model | Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-09-04 03:32:12,422 | INFO     | httpx | HTTP Request: HEAD https

Ready. 


#### 5.2 Cached Batch Processing(500 Items)

In [7]:
from datasets import load_dataset

# Silence the underlying httpx logger to keep the progress bar clean
logging.getLogger("httpx").setLevel(logging.WARNING)

PREPROCESSED_FILE = ROOT_DIR / "data" / "preprocessed_test_500.json"
NUM_EVAL_ITEMS = 500

@dataclass
class EvalItem:
    summary: str
    price: float
    title: str = ""

if not PREPROCESSED_FILE.exists():
    print(f"Loading raw test set and preprocessing {NUM_EVAL_ITEMS} items...")
    # Load test split (falls back to HuggingFace if 'test' is undefined)
    if 'test' not in globals():
        test_ds = load_dataset("ed-donner/items_lite", split="test")
        test = [item for item in test_ds]
    
    preprocessed_data = []
    for item in tqdm(test[:NUM_EVAL_ITEMS], desc="Preprocessing dataset"):
        summary_text = item["summary"] if isinstance(item, dict) else item.summary
        price_val = float(item["price"] if isinstance(item, dict) else item.price)
        title_text = item.get("title", summary_text[:40]) if isinstance(item, dict) else getattr(item, "title", summary_text[:40])

        rewritten_text = preprocessor.preprocess(summary_text)
        
        preprocessed_data.append({
            "original_summary": summary_text,
            "rewritten_summary": rewritten_text,
            "actual_price": price_val,
            "title": title_text
        })
        time.sleep(2.1)

    PREPROCESSED_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(PREPROCESSED_FILE, "w") as f:
        json.dump(preprocessed_data, f, indent=4)
        
    print(f"Saved preprocessed data to {PREPROCESSED_FILE}")
else:
    print(f"Found existing cached preprocessed file: {PREPROCESSED_FILE}")

with open(PREPROCESSED_FILE, "r") as f:
    raw_cached = json.load(f)

eval_data = [
    EvalItem(
        summary=d["original_summary"], 
        price=d["actual_price"], 
        title=d.get("title", "")
    )
    for d in raw_cached
]

print(f"Dataset prepared: {len(eval_data)} items loaded.")

Found existing cached preprocessed file: c:\user_personal_files\Projects\llm_engineering\price_is_right_ai\data\preprocessed_test_500.json
Dataset prepared: 500 items loaded.


##### 5.3.1 Individual Agent Benchmarks

In [39]:
# 1. Neural Network Agent
logging.getLogger().setLevel(logging.WARNING)
evaluate(
    function=lambda item: nn_agent.price(item),
    data=eval_data,
    size=NUM_EVAL_ITEMS,
    title="Neural Network Agent"
)

Eval Neural Network Agent: 100%|██████████| 500/500 [01:39<00:00,  5.01it/s]

$19 $40 $15 $21 $11 $88 $46 $14 $8 $5 $97 $49 $16 $2 $20 $1 $39 $13 $15 $241 $32 $24 $11 $117 $55 $229 $184 $15 $156 $50 $36 $19 $21 $50 $7 $162 $24 $22 $46 $2 $91 $28 $19 $37 $112 $4 $12 $1 $70 $2 $8 $37 $255 $57 $24 $42 $24 $26 $85 $20 $173 $43 $35 $35 $391 $26 $21 $318 $18 $82 $17 $6 $46 $41 $16 $3 $24 $1 $0 $3 $94 $25 $2 $53 $14 $28 $123 $34 $52 $121 $5 $8 $0 $3 $13 $14 $2 $36 $26 $227 $3 $45 $2 $82 $10 $26 $17 $250 $13 $101 $29 $15 $4 $38 $46 $48 $26 $14 $33 $112 $6 $51 $4 $27 $58 $53 $2 $35 $9 $12 $19 $17 $21 $1 $53 $3 $16 $34 $74 $22 $27 $174 $44 $43 $4 $11 $0 $237 $72 $12 $0 $96 $6 $42 $6 $152 $57 $8 $5 $17 $34 $11 $4 $0 $232 $1 $207 $14 $21 $11 $49 $5 $378 $20 $39 $9 $6 $17 $30 $31 $124 $2 $138 $14 $3 $24 $83 $12 $21 $15 $4 $8 $1 $45 $9 $41 $10 $3 $21 $12 $6 $8 $13 $13 $3 $8 $62 $128 $17 $63 $141 $129 $40 $7 $135 $7 $2 $7 $54 $22 $79 $18 $7 $49 $64 $78 $2 $3 $45 $4 $39 $3 

$60 $53 $11 $9 $10 $0 $15 $2 $0 $34 $43 $4 $77 $51 $72 $104 $1 $25 $27 $9 $94 $50 $15 $5 $7 $6 $47 $47 $16 $14 $8 $175 $20 $71 $1 $62 $34 $11 $3 $141 $14 $50 $107 $16 $18 $1 $6 $9 $9 $107 $5 $620 $6 $21 $23 $19 $24 $12 $4 $2 $206 $135 $7 $97 $5 $31 $18 $116 $180 $23 $15 $2 $172 $69 $88 $16 $164 $93 $2 $29 $11 $9 $45 $10 $69 $6 $10 $9 $265 $2 $88 $107 $11 $42 $52 $1 $5 $14 $80 $3 $6 $1 $2 $32 $9 $100 $9 $14 $82 $238 $7 $59 $131 $17 $7 $31 $4 $46 $636 $30 $11 $7 $7 $14 $31 $12 $3 $21 $7 $450 $12 $35 $13 $48 $83 $30 $2 $7 $10 $5 $81 $38 $17 $15 $11 $14 $11 $23 $33 $63 $5 $7 $40 $1 $32 $4 $21 $7 $61 $23 $1 $246 $53 $79 $30 $50 $8 $81 $10 $0 $38 $128 $226 $42 $39 $379 $153 $63 $2 $10 $85 $48 $15 $8 $19 $147 $29 $86 $28 $60 $490 $13 $7 $2 $51 $38 $3 $21 $24 $99 $13 $12 $418 $398 $34 $26 $60 $7 $1 $3 $20 $34 $5 $16 $114 $0 $101 $376 $541 $42 $4 $8 $1 $12 $17 $14 $22 $34 $1 $35 $47 $10 $46 $134 $0 $6 $8 $2 $31 $7 $27 $60 $126 $115 $68 $8 $5 $54 $5 $261 $47 $9 $5 $12 $20 $91 $1 $1 $110 $13 $16 

In [40]:
# 2. Specialist Agent (Local LLM)
logging.getLogger().setLevel(logging.WARNING)
evaluate(
    function=lambda item: specialist.price(item),
    data=eval_data,
    size=NUM_EVAL_ITEMS,
    title="Specialist Agent"
)

Eval Specialist Agent: 100%|██████████| 500/500 [05:52<00:00,  1.42it/s]

$10 $83 $21 $0 $20 $150 $54 $5 $4 $100 $43 $80 $0 $8 $19 $0 $21 $30 $41 $30 $34 $24 $16 $30 $174 $254 $196 $0 $100 $60 $20 $10 $100 $30 $2 $70 $80 $26 $74 $8 $110 $50 $5 $145 $90 $15 $12 $2 $85 $32 $15 $70 $325 $21 $17 $6 $13 $90 $69 $8 $137 $48 $20 $50 $399 $10 $40 $355 $2 $23 $13 $12 $60 $11 $0 $15 $26 $3 $3 $1 $30 $28 $5 $69 $7 $0 $43 $34 $30 $6 $8 $5 $0 $0 $2 $17 $4 $57 $100 $215 $0 $27 $3 $14 

$0 $19 $0 $280 $7 $0 $20 $29 $7 $52 $4 $0 $5 $5 $44 $47 $14 $111 $10 $12 $0 $39 $3 $1 $80 $99 $34 $17 $12 $0 $35 $2 $65 $10 $11 $31 $0 $249 $30 $0 $25 $18 $0 $221 $10 $3 $3 $13 $7 $40 $21 $49 $21 $6 $100 $0 $60 $16 $2 $1 $340 $7 $88 $20 $10 $3 $25 $5 $220 $7 $53 $0 $2 $17 $34 $9 $45 $5 $100 $25 $10 $8 $83 $13 $20 $3 $15 $1 $5 $1 $0 $30 $0 $13 $1 $10 $11 $7 $9 $23 $0 $23 $0 $40 $14 $61 $35 $150 $30 $4 $71 $5 $13 $13 $32 $6 $185 $0 $70 $47 $39 $80 $6 $9 $19 $12 $20 $0 $8 $59 $10 $25 $0 $0 $3 $29 $0 $19 $126 $17 $76 $40 $2 $40 $4 $12 $30 $0 $110 $26 $17 $0 $4 $0 $10 $30 $41 $89 $16 $152 $5 $40 $3 $55 $12 $12 $7 $124 $1 $8 $55 $11 $84 $11 $9 $8 $0 $0 $38 $398 $2 $16 $0 $18 $29 $40 $0 $1 $137 $290 $0 $48 $50 $0 $10 $170 $50 $70 $18 $3 $20 $84 $96 $10 $5 $15 $7 $20 $47 $2 $36 $6 $1 $40 $7 $0 $295 $0 $0 $55 $31 $84 $60 $1 $4 $19 $50 $2 $30 $0 $76 $26 $44 $107 $30 $9 $9 $390 $30 $16 $130 $3 $10 $24 $0 $20 $400 $10 $55 $15 $20 $60 $10 $255 $3 $0 $1 $496 $3 $25 $0 $6 $33 $65 $5 $4 $1 $53 $81 $2 

In [41]:
# 3. Frontier Agent (Dense + Cross-Encoder RAG)
logging.getLogger().setLevel(logging.WARNING)
evaluate(
    function=lambda item: frontier.price(item),
    data=eval_data,
    size=NUM_EVAL_ITEMS-250,
    title="Frontier Agent (RAG)"
)

Eval Frontier Agent (RAG): 100%|██████████| 250/250 [00:00<00:00, 43637.94it/s]

$10 $6 $1 $0 $2 $120 $54 $55 $4 $120 $2 $171 $5 $9 $34 $0 $36 $14 $65 $6 $51 $16 $46 $10 $87 $253 $70 $2 $146 $64 $15 $20 $89 $45 $35 $169 $26 $27 $49 $13 $157 $5 $10 $91 $30 $3 $25 $3 $78 $34 $23 $110 $125 $5 $47 $51 $1 $145 $37 $46 $76 $35 $66 $9 $144 $10 $0 $20 $19 $119 $18 $2 $11 $0 $40 $17 $21 $10 $3 $10 $30 $1 $15 $9 $2 $27 $205 $149 $35 $39 $23 $10 $10 $2 $0 $103 $0 $43 $20 $283 $125 $23 $12 $24 $0 $157 $12 $201 $14 $100 $20 $2 $14 $5 $34 $20 $0 $11 $59 $112 $9 $61 $25 $45 $39 $15 $5 $41 $6 $2 $0 $78 $20 $0 $85 $10 $80 $40 $7 $22 $6 $1 $15 $5 $1 $25 $35 $301 $0 $8 $10 $79 $23 $55 $6 $0 $56 $41 $75 $15 $10 $6 $4 $5 $467 $2 $137 $17 $26 $3 $29 $11 $170 $2 $17 $31 $17 $18 $85 $4 $65 $20 $205 $142 $10 

$23 $43 $22 $30 $4 $15 $8 $7 $61 $0 $70 $15 $30 $3 $1 $5 $218 $0 $29 $2 $22 $10 $40 $1 $46 $85 $202 $8 $13 $75 $10 $2 $4 $118 $35 $84 $20 $19 $72 $131 $20 $4 $5 $10 $68 $50 $4 $15 $74 $22 $50 $55 $2 $23 $55 $0 $19 $96 $2 $46 $85 $43 $10 $3 $48 
📊 Frontier Agent (RAG) Summary (N=250):
   • Mean Absolute Error (MAE):     $43.20
   • Median Absolute Error (MedAE): $20.27
   • Within 20% Tolerance (Hit%):  44.8%
   • Mean Squared Error (MSE):      5,390.07
   • R² Score:                      72.40%



##### 5.3.2 Full Ensemble Pipeline

In [43]:
# Mute the step-by-step agent logs so only the progress bar shows
logging.getLogger().setLevel(logging.WARNING)

class PassThroughPreprocessor:
    model_name = "Pass-Through (Raw Text)"
    def preprocess(self, text: str) -> str:
        return text

ensemble_eval = EnsembleAgent(
    specialist=specialist,
    frontier=frontier,
    neural_network=nn_agent,
    preprocessor=PassThroughPreprocessor()
)

evaluate(
    function=lambda item: ensemble_eval.price(item),
    data=eval_data,
    size=NUM_EVAL_ITEMS-250,
    title="Ensemble Agent Pipeline"
)

Eval Ensemble Agent Pipeline: 100%|██████████| 250/250 [08:31<00:00,  2.05s/it]

$4 $37 $11 $5 $9 $26 $52 $24 $2 $14 $40 $28 $6 $6 $15 $0 $31 $19 $44 $73 $1 $4 $10 $44 $110 $247 $143 $5 $133 $59 $22 $16 $9 $41 $13 $132 $44 $25 $57 $8 $124 $23 $1 $24 $71 $8 $17 $2 $78 $2 $17 $10 $227 $23 $30 $33 $11 $96 $30 $26 $121 $42 $42 $23 $295 $14 $19 $212 $4 $76 $16 $7 $37 $14 $12 $11 $23 $3 $0 $3 $22 $16 $4 $34 $7 $4 $128 $56 $10 $44 $13 $8 $4 $1 $4 $51 $2 $28 $36 $245 $49 $11 $3 $35 $2 $50 $1 $241 $11 $65 $22 $6 $7 $30 $4 $4 $9 $1 $47 $89 $10 $76 $7 $29 $1 

$21 $2 $26 $28 $37 $16 $41 $7 $0 $59 $4 $59 $11 $25 $14 $9 $43 $15 $15 $16 $19 $14 $257 $14 $7 $3 $60 $13 $46 $11 $55 $44 $20 $66 $2 $17 $11 $0 $2 $364 $4 $76 $17 $12 $5 $33 $1 $239 $8 $35 $10 $9 $18 $15 $6 $11 $10 $13 $45 $0 $18 $67 $7 $0 $4 $12 $5 $5 $36 $2 $7 $4 $16 $7 $7 $7 $88 $0 $16 $2 $15 $12 $62 $8 $56 $82 $166 $17 $6 $28 $4 $4 $4 $72 $21 $118 $13 $34 $57 $55 $40 $4 $0 $9 $31 $37 $2 $24 $64 $15 $31 $20 $1 $6 $12 $0 $23 $17 $8 $64 $61 $34 $44 $0 $30 
📊 Ensemble Agent Pipeline Summary (N=250):
   • Mean Absolute Error (MAE):     $34.75
   • Median Absolute Error (MedAE): $16.34
   • Within 20% Tolerance (Hit%):  56.0%
   • Mean Squared Error (MSE):      3,987.63
   • R² Score:                      79.58%



### 6. Deal Scraping & Autonomous Planning Agent (ReAct)

In [ ]:
# import importlib
# import src.agents.scanner_agent
# import src.agents.messaging_agent

# importlib.reload(src.agents.scanner_agent)
# importlib.reload(src.agents.messaging_agent)

<module 'src.agents.messaging_agent' from 'c:\\user_personal_files\\Projects\\llm_engineering\\price_is_right_ai\\src\\agents\\messaging_agent.py'>

#### 6.1 Scraping & Structured Extraction

In [46]:
from src.agents.scanner_agent import ScannerAgent

logger.info("Testing ScannerAgent...")
scanner = ScannerAgent()

# Test RSS parsing and concurrent scraping
raw_scraped = scanner.fetch_deals(known_urls=set())
print(f"✅ Raw items fetched from feeds: {len(raw_scraped)}")
if raw_scraped:
    print(f"Sample deal title: {raw_scraped[0].title}")

# Test Gemini Structured Output parsing
selection = scanner.scan(memory=[])
if selection and selection.deals:
    print(f"\n✅ Successfully extracted {len(selection.deals)} structured deals:")
    for idx, deal in enumerate(selection.deals, 1):
        print(f"  {idx}. [${deal.price:.2f}] {deal.product_description[:70]}...")
        print(f"    URL: {deal.url}")
else:
    print("⚠️ No valid deals returned by scanner.")

✅ Raw items fetched from feeds: 30
Sample deal title: Samsung M70H Series UN75M70HAFXZA 75" 4K HDR Mini LED UHD Smart TV for $550 + free shipping

✅ Successfully extracted 5 structured deals:
  1. [$644.50] This 43-inch floor-standing digital signage display features a durable...
    URL: https://www.dealnews.com/products/Jaszdot/Jaszdot-43-Floor-Standing-Digital-Signage-Display/529399.html?iref=rss-c142
  2. [$549.99] This 75-inch Samsung M70H Series smart TV features 4K resolution with ...
    URL: https://www.dealnews.com/products/Samsung/Samsung-M70-H-Series-UN75-M70-HAFXZA-75-4-K-HDR-Mini-LED-UHD-Smart-TV/530060.html?iref=rss-c142
  3. [$348.00] This Vizio 75-inch smart TV offers 4K UHD resolution at 3840 x 2160 pi...
    URL: https://www.dealnews.com/products/Vizio/75-4-K-LED-UHD-Smart-TV/498544.html?iref=rss-c142
  4. [$319.99] This certified refurbished Roborock Qrevo Pro robot vacuum features 70...
    URL: https://www.dealnews.com/Certified-Refurb-Roborock-Qrevo-Pro-Robot-Vac

#### 6.2 Push Notifications & Formatting

In [76]:
from src.agents.messaging_agent import MessagingAgent
from src.core.schemas import Deal, Opportunity

print("Testing MessagingAgent...")
messenger = MessagingAgent()

mock_deal = Deal(
    product_description="Corsair Vengeance LPX 32GB (2x16GB) DDR4 3200MHz C16 Desktop Memory Kit",
    price=49.99,
    url="https://example.com/corsair-deal"
)
mock_opportunity = Opportunity(
    deal=mock_deal,
    estimate=89.99,
    discount=40.00
)

# Test 1: Template alert (bypasses LLM, tests raw Pushover delivery)
print("Testing Ntfy template alert...")
messenger.alert(mock_opportunity)

# Test 2: LLM crafted push message via Groq
print("Testing Groq message crafting and delivery...")
messenger.notify(
    description=mock_deal.product_description,
    deal_price=mock_deal.price,
    estimated_true_value=mock_opportunity.estimate,
    url=mock_deal.url
)
print("✅ Check your Ntfy app for 2 notifications.")


Testing MessagingAgent...
Testing Ntfy template alert...
Testing Groq message crafting and delivery...
✅ Check your Ntfy app for 2 notifications.


### 7. Planning Agents

In [8]:
import importlib
import src.agents.deterministic_planning_agent
import src.agents.autonomous_planning_agent
import src.agents.frontier_agent
import src.agents.ensemble_agent
import src.agents.neural_network_agent
import src.agents.preprocessor_agent
import src.agents.specialist_agent
import src.agents.messaging_agent
import src.agents.scanner_agent

importlib.reload(src.agents.deterministic_planning_agent)
importlib.reload(src.agents.autonomous_planning_agent)
importlib.reload(src.agents.frontier_agent)
importlib.reload(src.agents.ensemble_agent)
importlib.reload(src.agents.neural_network_agent)
importlib.reload(src.agents.preprocessor_agent)
importlib.reload(src.agents.specialist_agent)
importlib.reload(src.agents.messaging_agent)
importlib.reload(src.agents.scanner_agent)

<module 'src.agents.scanner_agent' from 'c:\\user_personal_files\\Projects\\llm_engineering\\price_is_right_ai\\src\\agents\\scanner_agent.py'>

#### 7.1 Fixed Pipeline (DeterministicPlanningAgent)

In [88]:
import chromadb
from src.agents.deterministic_planning_agent import DeterministicPlanningAgent

# Initialize ChromaDB client pointing to project data folder
chroma_path = str(ROOT_DIR / "data" / "vectorstore")
client = chromadb.PersistentClient(path=chroma_path)
collection = client.get_or_create_collection("products")

planner = DeterministicPlanningAgent(
    collection=collection,
    model_url="http://localhost:11434" # Local Ollama endpoint for specialist
)

# Run end-to-end deterministic pipeline
print("Starting Deterministic Pipeline Run...")
best_opportunity = planner.plan(memory=[])

if best_opportunity:
    print(f"\n✅ Qualified Deal Found!")
    print(f"  • Product:  {best_opportunity.deal.product_description[:80]}...")
    print(f"  • Price:    ${best_opportunity.deal.price:.2f}")
    print(f"  • Estimate: ${best_opportunity.estimate:.2f}")
    print(f"  • Discount: ${best_opportunity.discount:.2f}")
else:
    print("\nℹ️ Run finished: No deals exceeded the DEAL_THRESHOLD.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2451.45it/s]


Starting Deterministic Pipeline Run...

✅ Qualified Deal Found!
  • Product:  ...
  • Price:    $15.00
  • Estimate: $137.26
  • Discount: $122.26


#### 7.2 ReAct Tool-Calling Loop (AutonomousPlanningAgent)

In [9]:
from src.agents.autonomous_planning_agent import AutonomousPlanningAgent

auto_planner = AutonomousPlanningAgent(
    collection=collection,
    model_url="http://localhost:11434"
)

# Run ReAct loop
print("Starting Autonomous Planning ReAct Run...")
auto_opportunity = auto_planner.plan(memory=[])

if auto_opportunity:
    print(f"\n✅ Autonomous Agent Surfaced Opportunity:")
    print(f"  • Price:    ${auto_opportunity.deal.price:.2f}")
    print(f"  • Estimate: ${auto_opportunity.estimate:.2f}")
    print(f"  • Margin:   ${auto_opportunity.discount:.2f}")
else:
    print("\nℹ️ Autonomous Agent completed run without surfacing deals.")

2026-09-04 03:34:22,757 | INFO     | Autonomous Planning Agent | [Autonomous Planning Agent] Autonomous Planning Agent is initializing
2026-09-04 03:34:22,758 | INFO     | Scanner Agent | [Scanner Agent] Initializing Scanner Agent...
2026-09-04 03:34:22,796 | INFO     | Specialist Agent | [Specialist Agent] Specialist Agent initialized: targeting local Ollama (specialist-pricer)
2026-09-04 03:34:22,798 | INFO     | Ensemble Agent | [Ensemble Agent] Initializing Ensemble Agent
2026-09-04 03:34:22,798 | INFO     | Frontier Agent | [Frontier Agent] Initializing Frontier Agent...
2026-09-04 03:34:22,832 | INFO     | Frontier Agent | [Frontier Agent] Loading BAAI Bi-Encoder globally on cuda...
2026-09-04 03:35:44,138 | INFO     | sentence_transformers.base.model | Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2190.50it/s]
2026-09-04 03:35:57,887 | INFO     | Frontier Agent | [Frontier Agent] Loading BAAI Cross-Encoder 

Starting Autonomous Planning ReAct Run...


2026-09-04 03:38:46,634 | INFO     | Autonomous Planning Agent | [Autonomous Planning Agent] Autonomous Planning agent is calling scanner
2026-09-04 03:39:03,076 | INFO     | Scanner Agent | [Scanner Agent] Concurrently scraped 30 new deals.
2026-09-04 03:39:03,079 | INFO     | Scanner Agent | [Scanner Agent] Requesting structured DealSelection from Gemini...
2026-09-04 03:39:47,480 | INFO     | Scanner Agent | [Scanner Agent] Successfully extracted 5 deals.
2026-09-04 03:39:47,483 | INFO     | Autonomous Planning Agent | [Autonomous Planning Agent] Preprocessing 5 deals via Groq...
2026-09-04 03:40:38,339 | INFO     | Autonomous Planning Agent | [Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
2026-09-04 03:40:38,341 | INFO     | Ensemble Agent | [Ensemble Agent] Running Ensemble Agent - preprocessing text
2026-09-04 03:40:45,475 | INFO     | Ensemble Agent | [Ensemble Agent] Pre-processed text using openai/gpt-oss-20b
2026-09-04 03:40:45,47


✅ Autonomous Agent Surfaced Opportunity:
  • Price:    $100.00
  • Estimate: $137.26
  • Margin:   $37.26


### 9. UI Dashboard Verification Streamlit

In [12]:
# Dynamically locate the project root and app.py
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
app_path = os.path.join(project_root, 'src', 'ui', 'app.py')

print(f"🚀 Launching Streamlit dashboard from: {app_path}")
print("👀 Look for the 'Local URL' or 'Network URL' link below and click it!\n")

# 2. Launch the Streamlit server directly from the notebook
import subprocess
import sys

# Launch Streamlit in the background
process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", app_path],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print(f"✅ Streamlit is running in the background (PID: {process.pid})")
print("Open your browser to http://localhost:8501")

🚀 Launching Streamlit dashboard from: c:\user_personal_files\Projects\llm_engineering\price_is_right_ai\src\ui\app.py
👀 Look for the 'Local URL' or 'Network URL' link below and click it!

✅ Streamlit is running in the background (PID: 27064)
Open your browser to http://localhost:8501
